# NLLB-200 Translation Benchmark: French → Kabyle

This notebook evaluates NLLB-200 translation quality using the **FLORES+ devtest** dataset.

## Metric Used
| Metric | What it Measures | Best For |
|--------|------------------|----------|
| **chrF++** | Character n-grams + word order | Morphologically rich languages (like Kabyle) ⭐ |

## Workflow
1. Download FLORES+ devtest (French & Kabyle) - ~1,000 sentences
2. Generate translations with NLLB model
3. Compute chrF++ score
4. Analyze results and inspect worst translations
5. Generate quality report with recommendations

In [2]:
# 1. Install Dependencies
%pip install sacrebleu datasets ctranslate2 transformers sentencepiece tqdm matplotlib

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached matplotlib-3.10.7-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached regex-2025.11.3-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl.metadata (6.4 kB)
  Using cached pillow-12.0.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached matp


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# 2. Load FLORES+ Dataset (French & Kabyle)
# First run: Downloads from HuggingFace and saves to local file
# Subsequent runs: Loads from local file (faster, no internet needed)

import os

# Local cache file path
FLORES_CACHE_FILE = "flores_fra_kab_pairs.txt"


def load_flores_from_cache(cache_file: str) -> tuple:
    """Load French-Kabyle pairs from local cache file."""
    src_sentences = []
    ref_sentences = []

    with open(cache_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and '\t' in line:
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    src_sentences.append(parts[0])
                    ref_sentences.append(parts[1])

    return src_sentences, ref_sentences


def save_flores_to_cache(src_sentences: list, ref_sentences: list, cache_file: str):
    """Save French-Kabyle pairs to local cache file."""
    with open(cache_file, 'w', encoding='utf-8') as f:
        for src, ref in zip(src_sentences, ref_sentences):
            # Replace any tabs/newlines in sentences to avoid format issues
            src_clean = src.replace('\t', ' ').replace('\n', ' ')
            ref_clean = ref.replace('\t', ' ').replace('\n', ' ')
            f.write(f"{src_clean}\t{ref_clean}\n")


# Check if cache exists
if os.path.exists(FLORES_CACHE_FILE):
    print(f"📂 Loading from local cache: {FLORES_CACHE_FILE}")
    src_sentences, ref_sentences = load_flores_from_cache(FLORES_CACHE_FILE)
    print(f"✅ Loaded {len(src_sentences)} sentence pairs from cache")

else:
    print("🌐 Cache not found. Downloading from HuggingFace...")
    print("⚠️ This is a gated dataset - you need to:")
    print("   1. Go to https://huggingface.co/datasets/openlanguagedata/flores_plus")
    print("   2. Accept the license agreement")
    print("   3. Have a valid HuggingFace token")

    from huggingface_hub import login
    from datasets import load_dataset

    # Login to HuggingFace (get your token from https://huggingface.co/settings/tokens)
    import os
    HF_TOKEN = os.getenv("HF_TOKEN")
    if HF_TOKEN:
        login(token=HF_TOKEN)
    else:
        print("HF_TOKEN not set - skipping HF login")

    print("\nDownloading FLORES+ devtest...")

    # Load French source sentences
    ds_fra = load_dataset("openlanguagedata/flores_plus",
                          "fra_Latn", split="devtest")
    # Load Kabyle reference sentences
    ds_kab = load_dataset("openlanguagedata/flores_plus",
                          "kab_Latn", split="devtest")

    # Extract sentences as lists
    src_sentences = list(ds_fra["text"])
    ref_sentences = list(ds_kab["text"])

    # Verify alignment
    assert len(src_sentences) == len(
        ref_sentences), "Sentence counts don't match!"

    # Save to cache for future use
    print(f"\n💾 Saving to local cache: {FLORES_CACHE_FILE}")
    save_flores_to_cache(src_sentences, ref_sentences, FLORES_CACHE_FILE)
    print(f"✅ Saved {len(src_sentences)} sentence pairs to cache")

# Display summary
print(f"\n{'='*50}")
print(f"📊 Dataset Summary:")
print(f"   French sentences:  {len(src_sentences)}")
print(f"   Kabyle sentences:  {len(ref_sentences)}")
print(f"{'='*50}")

print(f"\nExample:")
print(f"  FR:  {src_sentences[0]}")
print(f"  KAB: {ref_sentences[0]}")

📂 Loading from local cache: flores_fra_kab_pairs.txt
✅ Loaded 512 sentence pairs from cache

📊 Dataset Summary:
   French sentences:  512
   Kabyle sentences:  512

Example:
  FR:  Le Dr Ehud Ur, professeur de médecine à l'Université Dalhousie de Halifax (Nouvelle-Écosse) et président de la division clinique et scientifique de l'Association canadienne du diabète, a averti que la recherche en était encore à ses débuts.
  KAB: Amejjay Ehud Ur, aselmad n tujjya deg Tesdawit Dalhusi di Halifaks, Tikuṣt Tamaynut yerna d aselway n taẓunt taklinikt d tussnant n Tddukla Takanadyant n Waṭṭan n Skeṛ iεeggen-d d akken tagmi mazal-itt deg ussan-is imenza.


In [2]:
# 2b. Display Dataset for Manual Analysis
import pandas as pd
from IPython.display import display, HTML

# Create a DataFrame with both languages for manual inspection
dataset_df = pd.DataFrame({
    "idx": range(len(src_sentences)),
    "French (Source)": list(src_sentences),
    "Kabyle (Reference)": list(ref_sentences)
})

# Display settings
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 50)

print("="*80)
print("                    DATASET OVERVIEW: French → Kabyle")
print("="*80)
print(f"\nTotal sentences: {len(dataset_df)}")
print(
    f"Average French sentence length: {dataset_df['French (Source)'].str.len().mean():.1f} chars")
print(
    f"Average Kabyle sentence length: {dataset_df['Kabyle (Reference)'].str.len().mean():.1f} chars")

# Length ratio analysis
dataset_df["FR_len"] = dataset_df["French (Source)"].str.len()
dataset_df["KAB_len"] = dataset_df["Kabyle (Reference)"].str.len()
dataset_df["Length_Ratio"] = dataset_df["KAB_len"] / dataset_df["FR_len"]

print(
    f"Average length ratio (Kabyle/French): {dataset_df['Length_Ratio'].mean():.2f}")
print("\n" + "="*80)
print("                    FIRST 30 SENTENCE PAIRS")
print("="*80 + "\n")

# Show first 30 examples in a nice format
display(dataset_df[["idx", "French (Source)", "Kabyle (Reference)"]].head(30))

# Show some statistics
print("\n" + "="*80)
print("                    LAST 20 SENTENCE PAIRS")
print("="*80 + "\n")
display(dataset_df[["idx", "French (Source)", "Kabyle (Reference)"]].tail(20))

# Clean up temp columns
dataset_df = dataset_df.drop(columns=["FR_len", "KAB_len", "Length_Ratio"])

                    DATASET OVERVIEW: French → Kabyle

Total sentences: 512
Average French sentence length: 159.8 chars
Average Kabyle sentence length: 132.4 chars
Average length ratio (Kabyle/French): 0.84

                    FIRST 30 SENTENCE PAIRS



,idx,French (Source),Kabyle (Reference)
0,0,"Le Dr Ehud Ur, professeur de médecine à l'Université Dalhousie de Halifax (Nouvelle-Écosse) et p...","Amejjay Ehud Ur, aselmad n tujjya deg Tesdawit Dalhusi di Halifaks, Tikuṣt Tamaynut yerna d asel..."
1,1,"À l'instar d'autres experts, il se montre sceptique quant à la possibilité de guérir le diabète,...","Am yimusnawen niḍen, yella deg ul-is ccek ɣef ma yella dɣa d saḥ aṭṭan n skeṛ yesεa dwa-s, imi n..."
2,2,"Lundi, lors d'une émission de radio sur Sveriges Radio, en Suède, Sara Danius, secrétaire perman...","Deg wass n Warim, Sara Danyus, tamarayt tameɣlalt n Usmil Nubel n Tsekla deg Tesgrasnawit Taswid..."
3,3,"Danius a déclaré : « En ce moment, nous ne faisons rien. J'ai appelé et envoyé des courriels à s...","Tenna-d Danyus, ""Tura akka ur nxeddem acemma. Ssawleɣ yerna ceggεeɣ iznan iliktruniyen i yecrike..."
4,4,"Auparavant, le PDG de Ring, Jamie Siminoff, a fait remarquer qu'il a lancé l'entreprise lorsque ...","Zik yakkan, Anemhal n Ring, Ǧeymi Siminuff, yenna-d d akken takebbanit tebda mi ur yesli ara i a..."
5,5,M. Siminoff a déclaré que les ventes ont augmenté après son apparition en 2013 dans un épisode d...,Siminuff yenna-d d akken iznuzuyen ṭṭuqten-d seld mi d-iban deg uḥric n Cark Tank di 2013 anda i...
6,6,"Fin 2017, Siminoff est apparu sur la chaîne de télé-achat QVC.","Γer taggara n 2017, Siminuff iban-d deg umaṭṭaf n tiliẓṛi KFS."
7,7,"Ring a également réglé un procès avec une entreprise de sécurité concurrente, l'ADT Corporation.","Ring tsemsawi azmaẓ akked tkebbanit n tɣellist afna, Takebbanit ADT."
8,8,"Même si un vaccin expérimental semble jusqu'ici capable de réduire le taux de mortalité d'Ebola,...","Ɣas akken tban-d yiwet n tgezzayt n uɛraḍ i izemren ad isneqsen tamettant s waṭṭan n Ebula, er t..."
9,9,"Dans l'essai PALM, le ZMapp a servi de témoin, ce qui signifie que les scientifiques l'ont utili...","Deg uεraḍ BALM, ZMabb yella-d d tirmit tagejdant, aya yebɣa ad yini d akken imusnawen ssexdamen-..."



                    LAST 20 SENTENCE PAIRS



,idx,French (Source),Kabyle (Reference)
492,492,"On peut également dire qu’elle facilite la lecture, bien que l’écriture soit quelque peu compliq...","Mebla ccek daɣen d akken yessishel taɣuri, ɣas akken tira tewεeṛ akka ciṭ imi neḥwaǧ ad nẓer ma ..."
493,493,"La prononciation est relativement facile en italien, car la plupart des mots sont prononcés exac...",Asenṭeq yeshel ciṭ s Tṭelyanit imi tuget n wawalen ssenṭaqen-ten-id akken ten-ttarun.
494,494,"Les principales lettres à surveiller sont le c et le g, car leur prononciation varie en fonction...","Isekkilen igejdanen ilaq ad tḥadreḍ d c d g, imi asenṭeq-nsen yemgarad ɣef leḥsab n tiɣṛi i d-it..."
495,495,Veuillez également vous assurer de prononcer « r » et « rr » d’une manière distincte : « caro » ...,"Daɣen, tḥeq d akken tesneṭqeḍ-d r d rr s wudem amgarad: karu anamek-is aεziz, ma d karru anamek-..."
496,496,"Par conséquent, la lecture de cette introduction à la grammaire vous aiderait à en apprendre bea...","Ihi, taɣuri n tfelwit-agi n tjeṛṛumt ad k-tεiwen ad tlemdeḍ ugar ɣef tjeṛṛumt Tafarisit akked ad..."
497,497,"Il va sans dire que si vous connaissez une langue romane, il vous sera plus facile d'apprendre l...","Mabla ma nenna-d, ma yella tessneḍ tutlayt Taṛumit, ad yishil fell-ak ad tlemdeḍ Taburtugit."
498,498,"Comme la pollution lumineuse à leur époque n'était pas aussi préoccupant qu'aujourd'hui, ils son...","Imi asimes n tafat ur yelli d aɣbel am tura mi llan mechuren mliḥ, zgan-d s wudem imezgi di temd..."
499,499,La plupart des télescopes de recherche modernes sont d'énormes infrastructures placées dans des ...,Tuget n yitiliskuben n unadi atrar d arrumen anect-ilaten deg yimekwan ibeεden s tegnatin n unez...
500,500,"La contemplation des fleurs de cerisier, connue sous le nom de hanami, fait partie de la culture...","Imeẓri n ujuǧǧeg n tṛeḍlimt, yettwassnen s hanami, d aḥric n udles Ajabuni si tasut tis 8."
501,501,Le concept est venu de Chine où les fleurs de prunier étaient la fleur de prédilection.,Ugzu yekka-d si Ccinwa anda ajuǧǧeg n lbeṛquq llant d tijeǧǧigin n tefrant.


In [ ]:
# 3. Load NLLB Model (CTranslate2 version for efficiency)
import ctranslate2
from transformers import AutoTokenizer
import os
from huggingface_hub import snapshot_download

# Path to CTranslate2 model (download if not exists)
CT2_MODEL_PATH = "./nllb-200-3.3B-ct2"
MODEL_NAME = "facebook/nllb-200-3.3B"
CT2_MODEL_REPO = "JustFrederik/nllb-200-3.3B-ct2-float16"

# Download pre-converted model if needed
if not os.path.exists(CT2_MODEL_PATH):
    print("Downloading pre-converted CTranslate2 model (~6.5 GB)...")
    snapshot_download(repo_id=CT2_MODEL_REPO, local_dir=CT2_MODEL_PATH)
    print(f"✅ Downloaded to {CT2_MODEL_PATH}")
else:
    print(f"✅ Model already exists at {CT2_MODEL_PATH}")

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load CTranslate2 translator
print("Loading CTranslate2 model on CUDA...")
translator = ctranslate2.Translator(
    CT2_MODEL_PATH,
    device="cuda",
    compute_type="float16",
    inter_threads=1,
    intra_threads=4,
)
print("✅ Model loaded!")

d:\Documents\py\projects\chat_aqvayli\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Fetching 9 files:  11%|█         | 1/9 [00:01<00:12,  1.58s/it]

In [ ]:
# 4. Define Translation Function
from tqdm import tqdm

SRC_LANG = "fra_Latn"  # French
TGT_LANG = "kab_Latn"  # Kabyle


def translate_batch(texts: list, batch_size: int = 32) -> list:
    """
    Translate a list of texts from French to Kabyle.
    """
    tokenizer.src_lang = SRC_LANG
    all_translations = []

    # Convert to list if needed
    texts = list(texts)

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]

        # Tokenize
        source_tokens = [
            tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
            for text in batch
        ]

        # Translate
        target_prefix = [[TGT_LANG]] * len(batch)
        results = translator.translate_batch(
            source_tokens,
            target_prefix=target_prefix,
            beam_size=5,
            max_decoding_length=256,
        )

        # Decode
        for result in results:
            tokens = result.hypotheses[0]
            if tokens and tokens[0] == TGT_LANG:
                tokens = tokens[1:]
            translation = tokenizer.decode(
                tokenizer.convert_tokens_to_ids(tokens),
                skip_special_tokens=True
            )
            all_translations.append(translation)

    return all_translations


print("✅ Translation function defined!")

In [ ]:
# 5. Generate Translations (Step 1 of Benchmark)
print(f"Translating {len(src_sentences)} sentences from French to Kabyle...")
print("This may take a few minutes...\n")

# Generate hypotheses
hyp_sentences = translate_batch(src_sentences, batch_size=32)

print(f"\n✅ Generated {len(hyp_sentences)} translations")

# Save to file for reproducibility
with open("hypothesis_fr_kab.txt", "w", encoding="utf-8") as f:
    for sent in hyp_sentences:
        f.write(sent + "\n")
print("✅ Saved translations to hypothesis_fr_kab.txt")

# Show examples
print("\n" + "="*60)
print("Sample Translations:")
print("="*60)
for i in range(3):
    print(f"\n[{i+1}] FR:  {src_sentences[i]}")
    print(f"    REF: {ref_sentences[i]}")
    print(f"    HYP: {hyp_sentences[i]}")

## Step 2: Compute Metrics

We'll calculate:
- **chrF++**: Character-level F-score with word order (best for morphologically rich languages like Kabyle)

In [ ]:
# 6. Compute chrF++
import sacrebleu

# Ensure we have lists (not dataset columns)
refs = list(ref_sentences)
hyps = list(hyp_sentences)

# chrF++ (Character F-score with word order=2)
# This is the MOST RELIABLE metric for Kabyle due to its morphological richness
chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2)
print(f"{'='*50}")
print(f"chrF++ Score: {chrf.score:.2f}")
print(f"{'='*50}")
print(f"Details: {chrf}")

In [ ]:
# 7. Summary of Results
print("\n" + "="*60)
print("        BENCHMARK RESULTS: NLLB-200 French → Kabyle")
print("="*60)
print(f"\n  Dataset: FLORES+ devtest ({len(hyps)} sentences)")
print(f"\n  ┌{'─'*40}┐")
print(f"  │ {'Metric':<20} {'Score':>18} │")
print(f"  ├{'─'*40}┤")
print(f"  │ {'chrF++':<20} {chrf.score:>17.2f} │")
print(f"  └{'─'*40}┘")
print("\n  Interpretation:")
print("  • chrF++ is most reliable for Kabyle (morphologically rich)")
print("  • Measures character-level similarity with word order")
print("  • Compare against base NLLB-200 to measure improvement")
print("="*60)

## Step 3: Error Analysis

Inspecting the worst translations helps understand model failures.

In [ ]:
# 8. Analyze Worst Translations (by sentence-level chrF++)
import numpy as np

# Compute sentence-level chrF++ scores for analysis
from sacrebleu.metrics import CHRF
chrf_metric = CHRF(word_order=2)

sentence_scores = []
for hyp, ref in zip(hyps, refs):
    score = chrf_metric.sentence_score(hyp, [ref])
    sentence_scores.append(score.score)

# Create analysis dataframe
src = list(src_sentences)
analysis_data = [
    {
        "idx": i,
        "source": src[i],
        "reference": refs[i],
        "hypothesis": hyps[i],
        "chrf_score": sentence_scores[i]
    }
    for i in range(len(src))
]

# Sort by chrF++ score (ascending = worst first)
analysis_data.sort(key=lambda x: x["chrf_score"])

print("="*70)
print("                    WORST 10 TRANSLATIONS (by chrF++)")
print("="*70)

for i, item in enumerate(analysis_data[:10]):
    print(f"\n[{i+1}] chrF++ Score: {item['chrf_score']:.2f}")
    print(f"    FR:  {item['source'][:100]}..." if len(
        item['source']) > 100 else f"    FR:  {item['source']}")
    print(f"    REF: {item['reference'][:100]}..." if len(
        item['reference']) > 100 else f"    REF: {item['reference']}")
    print(f"    HYP: {item['hypothesis'][:100]}..." if len(
        item['hypothesis']) > 100 else f"    HYP: {item['hypothesis']}")
    print("-"*70)

In [ ]:
# 9. Score Distribution Analysis
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# chrF++ score distribution
axes[0].hist(sentence_scores, bins=50, edgecolor='black',
             alpha=0.7, color='steelblue')
axes[0].axvline(np.mean(sentence_scores), color='red',
                linestyle='--', label=f'Mean: {np.mean(sentence_scores):.2f}')
axes[0].axvline(np.median(sentence_scores), color='green',
                linestyle='--', label=f'Median: {np.median(sentence_scores):.2f}')
axes[0].set_xlabel('chrF++ Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('chrF++ Score Distribution')
axes[0].legend()

# Sentence length vs chrF++ score
src_lengths = [len(s.split()) for s in src]
axes[1].scatter(src_lengths, sentence_scores, alpha=0.5, s=10)
axes[1].set_xlabel('Source Sentence Length (words)')
axes[1].set_ylabel('chrF++ Score')
axes[1].set_title('Sentence Length vs Translation Quality')

plt.tight_layout()
plt.savefig('benchmark_analysis_fr_kab.png', dpi=150)
plt.show()

print(f"\n📊 Statistics:")
print(f"   Mean chrF++:   {np.mean(sentence_scores):.2f}")
print(f"   Median chrF++: {np.median(sentence_scores):.2f}")
print(f"   Std Dev:       {np.std(sentence_scores):.2f}")
print(f"   Min chrF++:    {np.min(sentence_scores):.2f}")
print(f"   Max chrF++:    {np.max(sentence_scores):.2f}")

In [ ]:
# 10. Save Full Results to CSV
import csv

output_file = "benchmark_results_fr_kab.csv"

with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["idx", "source_fr", "reference_kab",
                    "hypothesis", "chrf_score"])
    for item in analysis_data:
        writer.writerow([
            item["idx"],
            item["source"],
            item["reference"],
            item["hypothesis"],
            item["chrf_score"]
        ])

print(f"✅ Saved detailed results to {output_file}")
print(f"\n📁 Output files:")
print(f"   • hypothesis_fr_kab.txt - Model translations")
print(f"   • benchmark_results_fr_kab.csv - Full analysis with scores")
print(f"   • benchmark_analysis_fr_kab.png - Score distribution plots")

## Quality Report & Conclusion

In [ ]:
# ===== Utilities Functions =====

import os
import warnings
from huggingface_hub import login

# Use environment variable for Hugging Face token
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    warnings.warn("HF_TOKEN not set — huggingface_hub login skipped")

In [ ]:
# 12. Additional Recommendations from MT Research
print("=" * 80)
print("         RECOMMENDATIONS FROM MT RESEARCH & OTHER PROJECTS")
print("=" * 80)

recommendations = """
┌──────────────────────────────────────────────────────────────────────────────┐
│                    BEST PRACTICES FOR LOW-RESOURCE MT                        │
└──────────────────────────────────────────────────────────────────────────────┘

1️⃣  DATA AUGMENTATION STRATEGIES
   ─────────────────────────────────────────────────────────────────────────
   • Back-translation: Translate monolingual Kabyle text to French, use as 
     synthetic parallel data (Sennrich et al., 2016)
   • Paraphrasing: Generate variations of existing parallel sentences
   • Cross-lingual transfer: Leverage data from related Berber languages
     (Tamazight varieties: Tashelhit, Tarifit)
   
2️⃣  MODEL FINE-TUNING RECOMMENDATIONS
   ─────────────────────────────────────────────────────────────────────────
   • Start with NLLB-200 as base (already has Kabyle support)
   • Use LoRA or QLoRA for efficient fine-tuning on limited GPU memory
   • Train on domain-specific data if available (news, religious texts, etc.)
   • Consider continued pre-training on Kabyle monolingual data first
   
3️⃣  EVALUATION BEST PRACTICES (from WMT & FLORES papers)
   ─────────────────────────────────────────────────────────────────────────
   • chrF++ is preferred for morphologically rich languages (done ✓)
   • Perform human evaluation for final assessment
   • Use MQM (Multidimensional Quality Metrics) for error analysis
   • Report confidence intervals when possible
   
4️⃣  KNOWN ISSUES WITH NLLB FOR LOW-RESOURCE LANGUAGES
   ─────────────────────────────────────────────────────────────────────────
   • May produce "translationese" - grammatically correct but unnatural
   • Can hallucinate content not in source (check worst translations)
   • May copy source words when uncertain (especially for named entities)
   • Morphological errors common for agglutinative languages
   
5️⃣  COMPARISON BASELINES FROM LITERATURE
   ─────────────────────────────────────────────────────────────────────────
   From NLLB paper (Costa-jussà et al., 2022) for similar language pairs:
   
   │ Language Pair          │ chrF++  │ Notes                              │
   │────────────────────────│─────────│────────────────────────────────────│
   │ French → Berber (avg)  │ ~35-50  │ Varies by variety                  │
   │ Low-resource avg       │ ~30-45  │ NLLB-200 baseline                  │
   │ High-resource avg      │ ~55-70  │ For comparison                     │
   
   Note: Kabyle-specific benchmarks are limited in published literature.
   Your results contribute to the research on Berber language NLP!
   
6️⃣  RESOURCES FOR KABYLE NLP
   ─────────────────────────────────────────────────────────────────────────
   • NLLB FLORES dataset (using it ✓)
   • Tatoeba project for parallel sentences
   • Wikipedia in Kabyle (kab.wikipedia.org)
   • OASST multilingual dataset
   • Community-driven projects on GitHub
   
7️⃣  SUGGESTED NEXT STEPS
   ─────────────────────────────────────────────────────────────────────────
   □ Perform human evaluation on sample of 100 sentences
   □ Categorize errors (morphology, lexicon, syntax, meaning)
   □ Compare with English → Kabyle translation (different source language)
   □ Test on domain-specific texts (news, literature, technical)
   □ Collect additional parallel data for fine-tuning
   □ Try smaller NLLB variants (1.3B, distilled) for deployment
"""

print(recommendations)

# Save recommendations
with open("mt_recommendations_fr_kab.txt", "w", encoding="utf-8") as f:
    f.write(recommendations)
print("\n✅ Recommendations saved to mt_recommendations_fr_kab.txt")

In [ ]:
# 13. Memory Cleanup
import gc
import torch

del translator
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Memory cleaned up!")